<a href="https://colab.research.google.com/github/Coding-7allhh/YT_Automation/blob/main/All_Run_app_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
import shutil
import re

# ============================================
# 1) MULTI-FILE DOWNLOAD SYSTEM (ENVIRONMENT SECRETS)
# ============================================
!pip install --upgrade gdown > /dev/null

# Fallback links if the environment variable is not set
DEFAULT_URLS = [
    "https://colab.research.google.com/drive/1NY_YnlyrLlfyr2DersqOTKai1VL7Q55-"
]

# This looks for a secret named 'DRIVE_URLS'.
# In GitHub Actions, it will find it via os.environ.
# In Colab, you can set it in the 'Secrets' (key icon) sidebar.
env_urls = os.environ.get("DRIVE_URLS")
if env_urls:
    FILE_URLS = [url.strip() for url in env_urls.split(",")]
    print("✅ Using URLs from environment secrets.")
else:
    FILE_URLS = DEFAULT_URLS
    print("ℹ️ No 'DRIVE_URLS' secret found, using default list.")

LOCAL_PATH = "/content/downloaded_parts"

if os.path.exists(LOCAL_PATH):
    shutil.rmtree(LOCAL_PATH)
os.makedirs(LOCAL_PATH, exist_ok=True)

def get_drive_id(url):
    """Extracts ID from various Google Drive/Colab URL formats."""
    match = re.search(r"[-\w]{25,}", url)
    return match.group(0) if match else None

def download_source(url, target_dir):
    file_id = get_drive_id(url)
    print(f"🚀 Processing: {url[:50]}...")

    if "folders/" in url:
        cmd = ["gdown", "--no-check-certificate", "--folder", url, "-O", target_dir + "/"]
    elif file_id:
        cmd = ["gdown", "--no-check-certificate", f"https://drive.google.com/uc?id={file_id}", "-O", target_dir + "/"]
    else:
        cmd = ["gdown", "--no-check-certificate", url, "-O", target_dir + "/"]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Failed to download {url}: {result.stderr}")

# Download every link provided
for url in FILE_URLS:
    if url:
        download_source(url, LOCAL_PATH)

# ============================================
# 2) CONVERT & RUN AUTOMATICALLY
# ============================================
!pip install nbconvert > /dev/null

def run_workflow():
    downloaded_files = os.listdir(LOCAL_PATH)
    if not downloaded_files:
        print("❌ No files found in download directory.")
        return

    # Handle files without extensions
    for f in downloaded_files:
        path = os.path.join(LOCAL_PATH, f)
        if os.path.isfile(path) and "." not in f:
            new_path = path + ".ipynb"
            os.rename(path, new_path)

    # Convert ipynb to py
    notebooks = [os.path.join(LOCAL_PATH, f) for f in os.listdir(LOCAL_PATH) if f.endswith(".ipynb")]
    for nb in notebooks:
        print(f"🔄 Converting: {os.path.basename(nb)}")
        subprocess.run(["jupyter", "nbconvert", "--to", "python", nb])

    # Collect and run python scripts
    all_scripts = [os.path.join(LOCAL_PATH, f) for f in os.listdir(LOCAL_PATH) if f.endswith(".py")]

    if not all_scripts:
        print("❌ No runnable .py or .ipynb files found.")
        return

    all_scripts.sort()
    print(f"\n🔥 Running all scripts...\n")

    for script in all_scripts:
        name = os.path.basename(script)
        print(f"[EXEC] {name}")
        try:
            with open(script, 'r') as f:
                exec(f.read(), globals())
            print(f"[DONE] {name}")
        except Exception as e:
            print(f"[ERR ] {name}: {e}")

run_workflow()
print("\n✨ All Tasks Finished.")

ℹ️ No 'DRIVE_URLS' secret found, using default list.
🚀 Processing: https://colab.research.google.com/drive/1NY_YnlyrL...
🔄 Converting: 01_A_Long Fetcher 3.ipynb

🔥 Running all scripts...

[EXEC] 01_A_Long Fetcher 3.py
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package

KeyboardInterrupt: 